# Run ABI Benchmark

This notebook is the self-contained entry point for running the trained ABI amortizer on the exported ABI-vs-MCMC benchmark bank. It saves the same per-dataset and combined output tables as before, but everything now lives directly in the notebook.

In [1]:
from pathlib import Path
import importlib.util
import sys

project_dir = Path.cwd().resolve()
print('Python executable:', sys.executable)
print('Python version   :', sys.version.split()[0])
print('Project dir      :', project_dir)

for pkg in ['bayesflow', 'tensorflow', 'keras', 'numpy', 'pandas', 'scipy']:
    print(f'{pkg:10s}', importlib.util.find_spec(pkg) is not None)


Python executable: c:\Users\aieie\anaconda3\envs\bayesflow_env\python.exe
Python version   : 3.10.19
Project dir      : C:\Dati\Lavori\Aiello_Banerjee_2026\Code\ABI_poisson_regression\Simulation Experiments\ABI_vs_MCMC
bayesflow  True
tensorflow True
keras      True
numpy      True
pandas     True
scipy      True


In [3]:
from pathlib import Path

PROJECT_DIR = Path(r"C:/Dati/Lavori/Aiello_Banerjee_2026/Code/ABI_poisson_regression")
BENCHMARK_DIR = PROJECT_DIR / 'Simulation Experiments' / 'ABI_vs_MCMC' / 'datasets' / 'benchmark_bank_seed123_n100'
RESULTS_DIR = BENCHMARK_DIR / 'abi_results_all100'
CHECKPOINT = PROJECT_DIR / 'Training' / 'Checkpoints' / 'poisson_dagar.keras'
MCMC_REFERENCE_SAMPLER = 'dagar_poisson_boundary_mwg.cpp'

NUM_SAMPLES = 10_000
SEED = 123
SAVE_DRAWS = True
QUIET = False

# Set to a string like 'dataset_0000' or a list like ['dataset_0000', 'dataset_0001']
# to restrict the run. Leave as None for all datasets.
DATASET_IDS = None

# Leave as None for all datasets.
MAX_DATASETS = None

BENCHMARK_DIR, RESULTS_DIR, CHECKPOINT, MCMC_REFERENCE_SAMPLER


(WindowsPath('C:/Dati/Lavori/Aiello_Banerjee_2026/Code/ABI_poisson_regression/Simulation Experiments/ABI_vs_MCMC/datasets/benchmark_bank_seed123_n100'),
 WindowsPath('C:/Dati/Lavori/Aiello_Banerjee_2026/Code/ABI_poisson_regression/Simulation Experiments/ABI_vs_MCMC/datasets/benchmark_bank_seed123_n100/abi_results_all100'),
 WindowsPath('C:/Dati/Lavori/Aiello_Banerjee_2026/Code/ABI_poisson_regression/Training/Checkpoints/poisson_dagar.keras'),
 'dagar_poisson_boundary_mwg.cpp')

In [4]:
from __future__ import annotations

import os
import time
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import display

os.environ.setdefault('KERAS_BACKEND', 'tensorflow')
LOG2 = float(np.log(2.0))


def parse_dataset_ids(raw):
    if raw is None:
        return None
    if isinstance(raw, (list, tuple)):
        parts = [str(piece).strip() for piece in raw]
    else:
        parts = [piece.strip() for piece in str(raw).split(',')]
    parts = [piece for piece in parts if piece]
    return parts or None


def summarize_draws(samples: np.ndarray, truth: float, dataset_id: str, parameter: str) -> pd.DataFrame:
    q025, q50, q975 = np.quantile(samples, [0.025, 0.5, 0.975])
    mean_val = float(np.mean(samples))
    return pd.DataFrame([
        {
            'dataset_id': dataset_id,
            'parameter': parameter,
            'posterior_mean': mean_val,
            'posterior_sd': float(np.std(samples, ddof=1)),
            'posterior_median': float(q50),
            'lower_95': float(q025),
            'upper_95': float(q975),
            'truth': float(truth),
            'bias_mean': float(mean_val - truth),
            'abs_error_mean': float(abs(mean_val - truth)),
            'covered_95': int(q025 <= truth <= q975),
        }
    ])


def compute_auc(prob: np.ndarray, truth: np.ndarray) -> float:
    pos = np.flatnonzero(truth == 1)
    neg = np.flatnonzero(truth == 0)
    if pos.size == 0 or neg.size == 0:
        return float('nan')
    ranks = pd.Series(prob).rank(method='average').to_numpy()
    return float((ranks[pos].sum() - pos.size * (pos.size + 1) / 2.0) / (pos.size * neg.size))


def compute_average_precision(prob: np.ndarray, truth: np.ndarray) -> float:
    pos_total = int(np.sum(truth == 1))
    if pos_total == 0:
        return float('nan')
    order = np.argsort(-prob)
    truth_ord = truth[order]
    tp = np.cumsum(truth_ord == 1)
    precision = tp / np.arange(1, truth_ord.size + 1)
    return float(np.sum(precision[truth_ord == 1]) / pos_total)


def compute_brier(prob: np.ndarray, truth: np.ndarray) -> float:
    return float(np.mean((prob - truth) ** 2))


def compute_boundary_probabilities(eta_draws: np.ndarray, edge_z: np.ndarray, threshold: float = LOG2) -> np.ndarray:
    boundary_draws = (eta_draws[:, None] * edge_z[None, :]) > threshold
    return boundary_draws.mean(axis=0).astype(np.float32)


def load_dataset_npz(path: Path) -> dict[str, np.ndarray]:
    with np.load(path, allow_pickle=False) as data:
        return {key: data[key] for key in data.files}


def require_abi_dependencies():
    try:
        import bayesflow  # noqa: F401
    except ImportError as exc:
        raise ImportError(
            'The ABI notebook requires the ABI Python stack to be installed.\n'
            f'Current interpreter: {sys.executable}\n'
            f'Python version: {sys.version.split()[0]}\n\n'
            'Missing package: bayesflow\n\n'
            'Recommended fix on this machine:\n'
            '1. Create a dedicated conda environment with Python 3.11 or 3.12.\n'
            '2. Install bayesflow, tensorflow, keras, numpy, pandas, and scipy there.\n'
            '3. Run this notebook from that environment.\n\n'
            'Example:\n'
            '  conda create -n abi-benchmark python=3.11 -y\n'
            '  conda activate abi-benchmark\n'
            '  python -m pip install bayesflow tensorflow keras numpy pandas scipy'
        ) from exc

    try:
        import keras
    except ImportError as exc:
        raise ImportError(
            'The ABI notebook requires the ABI Python stack to be installed.\n'
            f'Current interpreter: {sys.executable}\n'
            f'Python version: {sys.version.split()[0]}\n\n'
            'Missing package: keras\n\n'
            'Recommended fix:\n'
            '  conda create -n abi-benchmark python=3.11 -y\n'
            '  conda activate abi-benchmark\n'
            '  python -m pip install bayesflow tensorflow keras numpy pandas scipy'
        ) from exc

    return bayesflow, keras


def set_global_seed(keras_module, seed: int) -> None:
    np.random.seed(seed)
    utils = getattr(keras_module, 'utils', None)
    if utils is not None and hasattr(utils, 'set_random_seed'):
        utils.set_random_seed(seed)


def build_conditions(dataset: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    obs = np.asarray(dataset['obs'], dtype=np.float32)
    n = int(dataset['N'])
    return {
        'obs': obs[np.newaxis, ...],
        'N': np.array([n], dtype=np.int32),
    }


def save_config_csv(path: Path, rows: Iterable[dict[str, object]]) -> None:
    df = pd.DataFrame(list(rows))
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def run_abi_benchmark_notebook(
    benchmark_dir: Path,
    results_dir: Path,
    checkpoint: Path,
    num_samples: int = 10_000,
    seed: int = 123,
    dataset_ids=None,
    max_datasets: int | None = None,
    save_draws: bool = True,
    quiet: bool = False,
):
    if num_samples <= 0:
        raise ValueError('num_samples must be positive.')
    if max_datasets is not None and max_datasets <= 0:
        raise ValueError('max_datasets must be positive when provided.')

    benchmark_dir = Path(benchmark_dir).resolve()
    checkpoint = Path(checkpoint).resolve()
    results_dir = Path(results_dir).resolve()

    manifest_path = benchmark_dir / 'benchmark_manifest.csv'
    datasets_dir = benchmark_dir / 'datasets'
    if not manifest_path.exists():
        raise FileNotFoundError(f'Missing benchmark manifest: {manifest_path}')
    if not datasets_dir.exists():
        raise FileNotFoundError(f'Missing datasets directory: {datasets_dir}')
    if not checkpoint.exists():
        raise FileNotFoundError(f'Missing ABI checkpoint: {checkpoint}')

    results_dir.mkdir(parents=True, exist_ok=True)
    per_dataset_dir = results_dir / 'per_dataset'
    per_dataset_dir.mkdir(parents=True, exist_ok=True)

    manifest = pd.read_csv(manifest_path)
    selected_ids = parse_dataset_ids(dataset_ids)
    if selected_ids is not None:
        manifest = manifest.loc[manifest['dataset_id'].isin(selected_ids)].copy()
    if max_datasets is not None:
        manifest = manifest.head(max_datasets).copy()
    if manifest.empty:
        raise ValueError('No datasets selected.')

    _, keras_module = require_abi_dependencies()
    set_global_seed(keras_module, seed)

    if not quiet:
        print(f'Loading ABI checkpoint: {checkpoint}')
    workflow = keras_module.saving.load_model(str(checkpoint))

    config_rows = [{
        'benchmark_dir': str(benchmark_dir),
        'results_dir': str(results_dir),
        'checkpoint': str(checkpoint),
        'mcmc_reference_sampler': MCMC_REFERENCE_SAMPLER,
        'num_samples': int(num_samples),
        'seed': int(seed),
        'dataset_ids': '' if selected_ids is None else ','.join(selected_ids),
        'max_datasets': '' if max_datasets is None else int(max_datasets),
        'save_draws': bool(save_draws),
    }]
    save_config_csv(results_dir / 'abi_config.csv', config_rows)

    all_parameter_summaries = []
    all_runtime_rows = []
    all_edge_metrics_rows = []

    for row_idx, row in enumerate(manifest.itertuples(index=False), start=1):
        dataset_id = str(row.dataset_id)
        file_name = getattr(row, 'file_name', f'{dataset_id}.npz')
        dataset_path = datasets_dir / file_name
        if not dataset_path.exists():
            raise FileNotFoundError(f'Missing dataset file for {dataset_id}: {dataset_path}')

        dataset = load_dataset_npz(dataset_path)
        conditions = build_conditions(dataset)
        m_val = float(np.asarray(dataset['M']).reshape(-1)[0])
        edge_i = np.asarray(dataset['edge_i'], dtype=np.int32)
        edge_j = np.asarray(dataset['edge_j'], dtype=np.int32)
        edge_z = np.asarray(dataset['edge_z'], dtype=np.float32)
        boundary_true = np.asarray(dataset['edge_boundary_true'], dtype=np.int32)

        if not quiet:
            print(f'[{row_idx}/{len(manifest)}] Running ABI for {dataset_id} (N={int(dataset["N"])}, edges={edge_i.size})')

        start = time.perf_counter()
        post_draws = workflow.sample(conditions=conditions, num_samples=num_samples)
        elapsed = time.perf_counter() - start

        beta0_draws = np.asarray(post_draws['beta'], dtype=np.float32)[0, :, 0]
        sigma2_w_draws = np.asarray(post_draws['sigma2_w'], dtype=np.float32)[0, :, 0]
        eta_raw_draws = np.asarray(post_draws['eta_raw'], dtype=np.float32)[0, :, 0]
        rho_draws = np.asarray(post_draws['rho'], dtype=np.float32)[0, :, 0]
        eta_draws = eta_raw_draws * m_val

        parameter_summaries = pd.concat([
            summarize_draws(beta0_draws, float(np.asarray(dataset['beta0_true']).reshape(-1)[0]), dataset_id, 'beta0'),
            summarize_draws(sigma2_w_draws, float(np.asarray(dataset['sigma2_w_true']).reshape(-1)[0]), dataset_id, 'sigma2_w'),
            summarize_draws(eta_raw_draws, float(np.asarray(dataset['eta_raw_true']).reshape(-1)[0]), dataset_id, 'eta_raw'),
            summarize_draws(eta_draws, float(np.asarray(dataset['eta_true']).reshape(-1)[0]), dataset_id, 'eta'),
            summarize_draws(rho_draws, float(np.asarray(dataset['rho_true']).reshape(-1)[0]), dataset_id, 'rho'),
        ], ignore_index=True)

        boundary_prob = compute_boundary_probabilities(eta_draws, edge_z, threshold=LOG2)
        boundary_median = (boundary_prob > 0.5).astype(np.int32)
        boundary_draws = (eta_draws[:, None] * edge_z[None, :]) > LOG2
        boundary_count_draws = boundary_draws.sum(axis=1).astype(np.int32)
        boundary_count_q = np.quantile(boundary_count_draws, [0.025, 0.975])

        edge_prob_df = pd.DataFrame({
            'edge_index_1based': np.arange(edge_i.size, dtype=np.int32) + 1,
            'node_i_1based': edge_i + 1,
            'node_j_1based': edge_j + 1,
            'edge_z': edge_z,
            'boundary_true': boundary_true,
            'dataset_id': dataset_id,
            'boundary_prob_abi': boundary_prob,
            'boundary_median_abi': boundary_median,
        })

        edge_metrics_df = pd.DataFrame([{
            'dataset_id': dataset_id,
            'edge_count': int(edge_i.size),
            'true_boundary_count': int(boundary_true.sum()),
            'posterior_boundary_count_mpm': int(boundary_median.sum()),
            'auroc': compute_auc(boundary_prob, boundary_true),
            'average_precision': compute_average_precision(boundary_prob, boundary_true),
            'brier': compute_brier(boundary_prob, boundary_true),
            'sensitivity_mpm': float(boundary_median[boundary_true == 1].mean()) if np.any(boundary_true == 1) else np.nan,
            'specificity_mpm': float((boundary_median[boundary_true == 0] == 0).mean()) if np.any(boundary_true == 0) else np.nan,
            'boundary_count_mean_draws': float(boundary_count_draws.mean()),
            'boundary_count_lower_95': float(boundary_count_q[0]),
            'boundary_count_upper_95': float(boundary_count_q[1]),
            'boundary_count_truth_in_95': int(boundary_count_q[0] <= boundary_true.sum() <= boundary_count_q[1]),
        }])

        runtime_df = pd.DataFrame([{
            'dataset_id': dataset_id,
            'elapsed_sec': elapsed,
            'n_saved_draws': int(beta0_draws.size),
            'num_samples_requested': int(num_samples),
            'seconds_per_saved_draw': float(elapsed / max(1, int(beta0_draws.size))),
            'seconds_per_1000_saved_draws': float(1000.0 * elapsed / max(1, int(beta0_draws.size))),
        }])

        dataset_out_dir = per_dataset_dir / dataset_id
        dataset_out_dir.mkdir(parents=True, exist_ok=True)
        parameter_summaries.to_csv(dataset_out_dir / f'{dataset_id}_parameter_summary.csv', index=False)
        edge_prob_df.to_csv(dataset_out_dir / f'{dataset_id}_edge_probabilities.csv', index=False)
        edge_metrics_df.to_csv(dataset_out_dir / f'{dataset_id}_edge_metrics.csv', index=False)
        runtime_df.to_csv(dataset_out_dir / f'{dataset_id}_runtime.csv', index=False)

        if save_draws:
            draws_df = pd.DataFrame({
                'dataset_id': dataset_id,
                'draw': np.arange(beta0_draws.size, dtype=np.int32) + 1,
                'beta0': beta0_draws,
                'sigma2_w': sigma2_w_draws,
                'eta_raw': eta_raw_draws,
                'eta': eta_draws,
                'rho': rho_draws,
            })
            draws_df.to_csv(dataset_out_dir / f'{dataset_id}_posterior_draws.csv', index=False)

        all_parameter_summaries.append(parameter_summaries)
        all_edge_metrics_rows.append(edge_metrics_df)
        all_runtime_rows.append(runtime_df)

    parameter_df = pd.concat(all_parameter_summaries, ignore_index=True)
    edge_metrics_df = pd.concat(all_edge_metrics_rows, ignore_index=True)
    runtime_df = pd.concat(all_runtime_rows, ignore_index=True)

    parameter_df.to_csv(results_dir / 'combined_parameter_summaries.csv', index=False)
    edge_metrics_df.to_csv(results_dir / 'combined_edge_metrics.csv', index=False)
    runtime_df.to_csv(results_dir / 'combined_runtime.csv', index=False)

    if not quiet:
        print(f'\nSaved ABI benchmark outputs to: {results_dir}')

    return {
        'parameter_df': parameter_df,
        'edge_metrics_df': edge_metrics_df,
        'runtime_df': runtime_df,
        'results_dir': results_dir,
        'benchmark_dir': benchmark_dir,
    }


In [5]:
run_outputs = run_abi_benchmark_notebook(
    benchmark_dir=BENCHMARK_DIR,
    results_dir=RESULTS_DIR,
    checkpoint=CHECKPOINT,
    num_samples=NUM_SAMPLES,
    seed=SEED,
    dataset_ids=DATASET_IDS,
    max_datasets=MAX_DATASETS,
    save_draws=SAVE_DRAWS,
    quiet=QUIET,
)

parameter_df = run_outputs['parameter_df']
edge_metrics_df = run_outputs['edge_metrics_df']
runtime_df = run_outputs['runtime_df']

display(parameter_df.head())
display(edge_metrics_df.head())
display(runtime_df.head())


INFO:bayesflow:Using backend 'tensorflow'


Loading ABI checkpoint: C:\Dati\Lavori\Aiello_Banerjee_2026\Code\ABI_poisson_regression\Training\Checkpoints\poisson_dagar.keras


[1/100] Running ABI for dataset_0000 (N=58, edges=161)


Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

[2/100] Running ABI for dataset_0001 (N=85, edges=242)


...

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]


Saved ABI benchmark outputs to: C:\Dati\Lavori\Aiello_Banerjee_2026\Code\ABI_poisson_regression\Simulation Experiments\ABI_vs_MCMC\datasets\benchmark_bank_seed123_n100\abi_results_all100


,dataset_id,parameter,posterior_mean,posterior_sd,posterior_median,lower_95,upper_95,truth,bias_mean,abs_error_mean,covered_95
0,dataset_0000,beta0,-0.709619,0.072406,-0.711638,-0.807672,-0.617500,-0.685540,-0.024079,0.024079,1
1,dataset_0000,sigma2_w,0.391629,0.174728,0.348860,0.187546,0.867136,0.181137,0.210493,0.210493,0
2,dataset_0000,eta_raw,0.319897,0.243645,0.258186,0.014033,0.933616,0.569412,-0.249516,0.249516,1
3,dataset_0000,eta,0.207372,0.157942,0.167368,0.009097,0.605212,0.369119,-0.161747,0.161747,1
4,dataset_0000,rho,0.289253,0.181949,0.257162,0.023077,0.706689,0.186174,0.103079,0.103079,1


,dataset_id,edge_count,true_boundary_count,posterior_boundary_count_mpm,auroc,average_precision,brier,sensitivity_mpm,specificity_mpm,boundary_count_mean_draws,boundary_count_lower_95,boundary_count_upper_95,boundary_count_truth_in_95
0,dataset_0000,161,33,0,1.0,1.0,0.111054,0.000000,1.000000,12.8478,0.0,75.000,1
1,dataset_0001,242,121,76,1.0,1.0,0.136002,0.628099,1.000000,71.0112,0.0,117.000,0
2,dataset_0002,490,8,4,1.0,1.0,0.017620,0.500000,1.000000,30.4936,0.0,157.000,1
3,dataset_0003,381,158,163,1.0,1.0,0.016915,1.000000,0.977578,160.9913,103.0,189.000,1
4,dataset_0004,638,148,39,1.0,1.0,0.105438,0.263514,1.000000,55.3416,0.0,194.025,1


,dataset_id,elapsed_sec,n_saved_draws,num_samples_requested,seconds_per_saved_draw,seconds_per_1000_saved_draws
0,dataset_0000,0.773797,10000,10000,0.000077,0.077380
1,dataset_0001,0.768197,10000,10000,0.000077,0.076820
2,dataset_0002,0.766981,10000,10000,0.000077,0.076698
3,dataset_0003,0.761138,10000,10000,0.000076,0.076114
4,dataset_0004,0.762078,10000,10000,0.000076,0.076208
